# Stage A — Inspect the pre-fitted lens

**Time:** about 5 minutes. **GPU:** not needed — leave the runtime on CPU.

### What a "lens" is, in one paragraph

The Jacobian lens is a set of matrices, one per layer of the model. Each matrix translates what's happening inside that layer into "which words is the model being pushed toward saying." It has to be *fitted* — computed once from the model by running some text through it. Anthropic published a pre-fitted lens for this exact model, so you may not need to fit your own.

### What we're checking, and why it matters

There are two recipes for fitting. The published code defaults to one; the paper used the other. Appendix A.7 says the paper's version is less noisy, and we measured the two to differ by 2.4%.

The saved lens file does **not** record which recipe was used. But it can be inferred: the fitter requires every layer it reads from to sit *below* the layer it targets. So if the lens reads from layers all the way up to the second-to-last, the target must have been the very last layer — the code's default, not the paper's.

## Cell 1 — Install

In [ ]:
!pip -q install -U transformers huggingface_hub
!git clone -q --depth 1 https://github.com/anthropics/jacobian-lens.git
%cd /content/jacobian-lens
!pip -q install -e .

import jlens
print("jlens imported OK")

## Cell 2 — See what's in the lens repository

Lists the files without downloading them. You're looking for a filename that mentions this model, and any README or config.

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("neuronpedia/jacobian-lens")
print(f"{len(files)} files in the repo:\n")
for f in sorted(files):
    print("  ", f)

## Cell 3 — Download and inspect the lens

Set `LENS_FILE` to whichever filename from Cell 2 corresponds to Qwen3.5-4B. If the names are unclear, paste Cell 2's output back into the chat before running this.

In [ ]:
LENS_FILE = "qwen3.5-4b/lens.pt"      # <-- edit to match Cell 2's listing

import torch
from huggingface_hub import hf_hub_download

path = hf_hub_download("neuronpedia/jacobian-lens", filename=LENS_FILE)
ckpt = torch.load(path, map_location="cpu", weights_only=True)

print("keys stored in the file:", sorted(ckpt.keys()))
print()
print("d_model       :", ckpt["d_model"])
print("n_prompts     :", ckpt["n_prompts"], " (how much text it was fitted on)")
print("source_layers :", ckpt["source_layers"])
print("matrix shape  :", tuple(next(iter(ckpt["J"].values())).shape))
print("stored dtype  :", next(iter(ckpt["J"].values())).dtype)

## Cell 4 — Work out which recipe was used

This reads the model's configuration to find out how many layers it has, then applies the inference described at the top.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3.5-4B")
n_layers = cfg.num_hidden_layers
max_source = max(ckpt["source_layers"])

print(f"model has {n_layers} layers (indices 0 to {n_layers - 1})")
print(f"lens reads from layers {min(ckpt['source_layers'])} to {max_source}")
print()

if max_source == n_layers - 2:
    print(">>> TARGET WAS THE FINAL LAYER — the code default, NOT the paper's recipe.")
    print(">>> A lens reading up to the second-to-last layer can only have")
    print(">>> targeted the last one. See the decision note below.")
elif max_source <= n_layers - 3:
    print(">>> CONSISTENT with the paper's recipe (penultimate target).")
    print(">>> Not proof — they may simply have fitted fewer layers — but the")
    print(">>> final-layer default is ruled out.")
else:
    print(">>> UNEXPECTED. Paste this output into the chat before continuing.")

print()
print(f"d_model = {ckpt['d_model']}")
print(f"If you fit your own: ~{-(-ckpt['d_model'] // 8)} backward passes per prompt at dim_batch=8.")

## Cell 5 — Check the repo for any written documentation

If the authors documented the fitting recipe anywhere, it would be here. This is the direct evidence; Cell 4 is only an inference.

In [ ]:
from huggingface_hub import hf_hub_download

for name in ["README.md", "config.json", "lens_config.json"]:
    try:
        p = hf_hub_download("neuronpedia/jacobian-lens", filename=name)
        print(f"===== {name} =====")
        print(open(p).read()[:3000])
        print()
    except Exception as e:
        print(f"{name}: not present ({type(e).__name__})")

---
## Stop here and report back

Paste into the chat:

1. Cell 2's file listing
2. Cell 3's output (`d_model`, `n_prompts`, `source_layers`)
3. Cell 4's verdict
4. Anything Cell 5 found

**Do not start Stage B yet.** What the verdict says determines whether you use this lens or fit your own, and fitting is the expensive step — worth being sure before spending GPU time on it.

### What each verdict means

| Verdict | What happens next |
|---|---|
| Consistent with the paper's recipe | Use this lens. Straight to Stage B. |
| Target was the final layer | A choice: use it and record the deviation in the paper, or fit your own (~15–30 min GPU, some memory risk). Control A is already partial by construction, so this deviation is unlikely to be what decides the paper. |
| Unexpected | Don't guess. Paste it and we'll work it out. |